# SeaweedFS + Apache Iceberg 1.12 + Spark 4.2

A guided tour of the stack in this repo.

* Sections 1-7 are the classic Iceberg concepts.
* Sections 8-11 are what is **new**: Iceberg 1.12 behaviour and the Spark 4.2
  SQL surface.
* Sections 12-15 cover maintenance, rollback, the metadata tables and
  SeaweedFS' own Iceberg REST catalog. Section 16 lists the gaps you will hit.

| Catalog | Type | Metadata lives in |
|---|---|---|
| `my_catalog` (default) | `JdbcCatalog` | Postgres |
| `seaweed_catalog` | Iceberg REST catalog | SeaweedFS itself, on `:8181` |

For both, the actual data and metadata *files* live in SeaweedFS, reached
through Iceberg's `S3FileIO`.

Every code cell below was executed against the real stack before being written
down.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import month, to_date

spark = SparkSession.builder.appName("iceberg-seaweedfs-demo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Spark   :", spark.version)
print("Catalogs:", sorted(row[0] for row in spark.sql("SHOW CATALOGS").collect()))

:: loading settings :: file = /opt/spark/conf/ivysettings.xml


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/spark/.ivy2.5.2/cache
The jars for the packages stored in: /home/spark/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.2_2.13 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d787908f-9a45-4b4b-9b4c-325fa87f6f38;1.0
	confs: [default]


	found org.apache.iceberg#iceberg-spark-runtime-4.2_2.13;1.12.0-SNAPSHOT in apache-snapshots


	found org.apache.iceberg#iceberg-aws-bundle;1.12.0-SNAPSHOT in apache-snapshots


	found org.postgresql#postgresql;42.7.13 in central


...

	[SUCCESSFUL ] org.postgresql#postgresql;42.7.13!postgresql.jar (528ms)
downloading https://repo1.maven.org/maven2/org/checkerframework/checker-qual/3.55.1/checker-qual-3.55.1.jar ...
	[SUCCESSFUL ] org.checkerframework#checker-qual;3.55.1!checker-qual.jar (174ms)
:: resolution report :: resolve 14531ms :: artifacts dl 213118ms
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   4   |   4   |   4   |   0   ||   4   |   4   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-d787908f-9a45-4b4b-9b4c-325fa87f6f38
	confs: [default]


	4 artifacts copied, 0 already retrieved (117408kB/56ms)
26/10/02 12:19:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/02 12:19:10 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


Spark   : 4.2.0


Catalogs: ['spark_catalog']


In [2]:
# The first run of this notebook resolves spark.jars.packages through Ivy,
# downloading ~115 MB of Iceberg/JDBC jars. They are cached in ~/.ivy2, so
# later runs start immediately.
spark.sql("SELECT 1 AS ok").show()

+---+
| ok|
+---+
|  1|
+---+



## 0.1 Where each catalog points

In [3]:
for name in ("my_catalog", "seaweed_catalog"):
    print(f"--- {name} ---")
    for suffix in ("type", "catalog-impl", "warehouse", "uri", "io-impl", "s3.endpoint"):
        try:
            print(f"  {suffix:14} = {spark.conf.get('spark.sql.catalog.' + name + '.' + suffix)}")
        except Exception:
            pass

--- my_catalog ---
  catalog-impl   = org.apache.iceberg.jdbc.JdbcCatalog
  warehouse      = s3://iceberg
  uri            = jdbc:postgresql://db:5432/icebergcat
  io-impl        = org.apache.iceberg.aws.s3.S3FileIO
  s3.endpoint    = http://seaweedfs:8333
--- seaweed_catalog ---
  type           = rest
  warehouse      = s3://iceberg-catalog
  uri            = http://seaweedfs:8181
  io-impl        = org.apache.iceberg.aws.s3.S3FileIO
  s3.endpoint    = http://seaweedfs:8333


`my_catalog` keeps table pointers in Postgres while every metadata and data file
is an object in SeaweedFS. `seaweed_catalog` speaks the Iceberg REST protocol
directly to SeaweedFS, which is what lets Trino, DuckDB, RisingWave, Dremio,
Doris and Lakekeeper read the same tables without sharing a database.

## 1. Create the table

In [4]:
# Start from a clean slate so the notebook can be re-run.
for leftover in ("spy_table", "spy_data"):
    spark.sql("DROP TABLE IF EXISTS my_catalog." + leftover)

spark.sql("""
CREATE TABLE my_catalog.spy_data (
    Date       date,
    Open       double,
    High       double,
    Low        double,
    Close      double,
    Adj_Close  double,
    Volume     integer)
USING iceberg
""")

spark.sql("DESCRIBE TABLE my_catalog.spy_data").show()

+---------+---------+-------+
| col_name|data_type|comment|
+---------+---------+-------+
|     Date|     date|   NULL|
|     Open|   double|   NULL|
|     High|   double|   NULL|
|      Low|   double|   NULL|
|    Close|   double|   NULL|
|Adj_Close|   double|   NULL|
|   Volume|      int|   NULL|
+---------+---------+-------+



`CREATE TABLE` wrote the first `metadata.json` into the `iceberg` bucket that
`weed mini` created on boot. You never create that bucket by hand:

```bash
docker compose exec seaweedfs weed shell -master=seaweedfs:9333 -command="s3.bucket.list"
```

Browse the object store at <http://localhost:18888> (the filer UI).

## 2. Load the data

In [5]:
df = spark.read.csv("/home/spark/data/spy_data.csv", header=True, inferSchema=True)

# inferSchema infers a timestamp for Date; Iceberg wants a real date.
df = df.withColumn("Date", to_date("Date"))

first_decade = df.filter("Date >= '1993-01-01' AND Date <= '2003-12-31'")
print("rows:", first_decade.count())

first_decade.writeTo("my_catalog.spy_data").append()

spark.sql("SELECT * FROM my_catalog.spy_data ORDER BY Date").show(5)

rows: 2753


+----------+--------+--------+--------+--------+------------------+-------+
|      Date|    Open|    High|     Low|   Close|         Adj_Close| Volume|
+----------+--------+--------+--------+--------+------------------+-------+
|1993-01-29|43.96875|43.96875|   43.75| 43.9375|25.122343063354492|1003200|
|1993-02-01|43.96875|   44.25|43.96875|   44.25| 25.30103302001953| 480500|
|1993-02-02|44.21875|  44.375|  44.125|44.34375|25.354625701904297| 201300|
|1993-02-03|44.40625|44.84375|  44.375| 44.8125|25.622648239135742| 529400|
|1993-02-04|44.96875|45.09375|44.46875|    45.0| 25.72984504699707| 531500|
+----------+--------+--------+--------+--------+------------------+-------+
only showing top 5 rows


## 3. Schema evolution

In [6]:
spark.sql("ALTER TABLE my_catalog.spy_data RENAME TO spy_table")

# Add a column at a specific position in the schema.
spark.sql("ALTER TABLE my_catalog.spy_table ADD COLUMN Month int AFTER Date")
spark.sql("UPDATE spy_table SET Month = MONTH(Date)")

spark.sql("DESCRIBE TABLE my_catalog.spy_table").show()

+---------+---------+-------+
| col_name|data_type|comment|
+---------+---------+-------+
|     Date|     date|   NULL|
|    Month|      int|   NULL|
|     Open|   double|   NULL|
|     High|   double|   NULL|
|      Low|   double|   NULL|
|    Close|   double|   NULL|
|Adj_Close|   double|   NULL|
|   Volume|      int|   NULL|
+---------+---------+-------+



The `ALTER TABLE ADD COLUMN` is metadata-only and instant. The `UPDATE` is not:
under the default copy-on-write mode Iceberg rewrites the data files it has to
change. The superseded files stay reachable through time travel, which is the
next section.

## 4. Partition evolution and hidden partitioning

In [7]:
spark.sql("ALTER TABLE my_catalog.spy_table ADD PARTITION FIELD Month")

df = df.withColumn("Month", month(df.Date))
next_two_decades = df.filter("Date >= '2004-01-01' AND Date <= '2023-12-31'")
next_two_decades.writeTo("my_catalog.spy_table").append()

spark.sql("""
SELECT partition, count(*) AS files, sum(record_count) AS records
FROM my_catalog.spy_table.files
GROUP BY partition
ORDER BY files DESC
LIMIT 8
""").show(truncate=40)

+---------+-----+-------+
|partition|files|records|
+---------+-----+-------+
|     {12}|    1|    401|
|      {9}|    1|    389|
|      {6}|    1|    407|
|     {10}|    1|    416|
|      {5}|    1|    400|
|   {NULL}|    1|   2753|
|      {7}|    1|    400|
|     {11}|    1|    390|
+---------+-----+-------+



In [8]:
# The query never mentions partitioning. Iceberg prunes for us: rows before
# 2004 live in unpartitioned files, rows from 2004 live under month=N/.
spark.sql("""
SELECT Date, Month, Close
FROM my_catalog.spy_table
WHERE Date >= '2000-01-01' AND Date <= '2006-01-01'
ORDER BY Date
""").show(5)

+----------+-----+--------+
|      Date|Month|   Close|
+----------+-----+--------+
|2000-01-03|    1|145.4375|
|2000-01-04|    1|  139.75|
|2000-01-05|    1|   140.0|
|2000-01-06|    1|  137.75|
|2000-01-07|    1|  145.75|
+----------+-----+--------+
only showing top 5 rows


Walk `iceberg / my_catalog / spy_table / data` in the filer UI at
<http://localhost:18888>. You will see loose parquet files for the pre-2004 rows
next to `month=1/`, `month=2/`, ... directories for everything written after the
partition field was added. Two partition specs coexist in one table - this is
**partition evolution**, and the fact that queries never have to care is
**hidden partitioning**.

## 5. Time travel

In [9]:
spark.sql("""
SELECT snapshot_id, operation, summary['added-records'] AS added
FROM my_catalog.spy_table.snapshots
""").show(truncate=50)

+-------------------+---------+-----+
|        snapshot_id|operation|added|
+-------------------+---------+-----+
|2209018314294998135|   append| 2753|
|8755055807240082631|overwrite| 2753|
|8743253262251600771|   append| 4820|
+-------------------+---------+-----+



In [10]:
snapshots = spark.sql(
    "SELECT snapshot_id FROM my_catalog.spy_table.snapshots ORDER BY committed_at"
).collect()
oldest = snapshots[0]["snapshot_id"]

print("rows at oldest snapshot:", spark.sql(
    "SELECT count(*) FROM my_catalog.spy_table VERSION AS OF " + str(oldest)).collect()[0][0])
print("rows at current head  :", spark.sql(
    "SELECT count(*) FROM my_catalog.spy_table").collect()[0][0])
print("rows in 2019, as of now:", spark.sql(
    "SELECT count(*) FROM my_catalog.spy_table WHERE Year(Date) = 2019").collect()[0][0])

rows at oldest snapshot: 2753
rows at current head  : 7573
rows in 2019, as of now: 252


Every snapshot stays addressable by id, and by timestamp when you remember when
rather than which:

```sql
SELECT * FROM my_catalog.spy_table VERSION AS OF 337209422568253609;
SELECT * FROM my_catalog.spy_table TIMESTAMP AS OF '2022-07-25 12:15:00';
```

The oldest snapshot predates the `Month` partition field, so the same table
returns rows in two different layouts depending on which snapshot you ask for.
Section 13 rolls back for real.

## 6. Row-level `UPDATE` and `DELETE`

In [11]:
spark.sql("SELECT Date, Month, Close FROM my_catalog.spy_table ORDER BY Date DESC LIMIT 5").show()

spark.sql("DELETE FROM my_catalog.spy_table WHERE Year(Date) = 2023")
spark.sql("UPDATE my_catalog.spy_table SET Volume = Volume / 1000 WHERE Year(Date) = 2004")

spark.sql("""
SELECT Year(Date) AS yr, count(*) AS rows, sum(Volume) AS volume
FROM my_catalog.spy_table
WHERE Year(Date) IN (2003, 2004, 2022, 2023)
GROUP BY Year(Date)
ORDER BY yr
""").show()

+----------+-----+-----------------+
|      Date|Month|            Close|
+----------+-----+-----------------+
|2023-02-24|    2|396.3800048828125|
|2023-02-23|    2|400.6600036621094|
|2023-02-22|    2|398.5400085449219|
|2023-02-21|    2|399.0899963378906|
|2023-02-17|    2| 407.260009765625|
+----------+-----+-----------------+



+----+----+-----------+
|  yr|rows|     volume|
+----+----+-----------+
|2003| 252|10303214100|
|2004| 252|   10794378|
|2022| 251|23784662100|
+----+----+-----------+



Spark 4.2 reports row-level operation metrics for Data Source V2
(`numUpdatedRows`, `numDeletedRows`, `numCopiedRows`). They appear in `EXPLAIN`
and in the SQL tab of the Web UI:

```sql
EXPLAIN EXTENDED DELETE FROM my_catalog.spy_table WHERE Date = '2022-05-02'
```

## 7. Merge-on-read deletes

In [12]:
# Copy-on-write is the default: a DELETE rewrites whole data files.
# Merge-on-read instead writes small "position delete" files.
spark.sql("ALTER TABLE my_catalog.spy_table "
          "SET TBLPROPERTIES ('write.delete.mode' = 'merge-on-read')")

before = spark.sql("SELECT count(*) FROM my_catalog.spy_table").collect()[0][0]
spark.sql("DELETE FROM my_catalog.spy_table WHERE Year(Date) = 2020")
after = spark.sql("SELECT count(*) FROM my_catalog.spy_table").collect()[0][0]
print("before=", before, " after=", after)

before= 7536  after= 7283


## 8. NEW - Iceberg branches and tags

A branch is a named, independently movable pointer into the snapshot history; a
tag pins one snapshot. Both are metadata-only, so they cost nothing until you
write to them, and they give you zero-copy isolation for experiments and audits.

In [13]:
spark.sql("ALTER TABLE my_catalog.spy_table CREATE BRANCH IF NOT EXISTS audit_branch")
spark.sql("ALTER TABLE my_catalog.spy_table CREATE TAG IF NOT EXISTS release_2023")

spark.sql("""
INSERT INTO my_catalog.spy_table.branch_audit_branch
  (Date, Open, High, Low, Close, Adj_Close, Volume, Month)
VALUES (DATE '2024-01-02', 470.0, 472.0, 468.0, 471.0, 471.0, 4000000, 1)
""")

spark.sql("SELECT * FROM my_catalog.spy_table.refs").show(truncate=50)

+------------+------+-------------------+-----------------------+---------------------+----------------------+
|        name|  type|        snapshot_id|max_reference_age_in_ms|min_snapshots_to_keep|max_snapshot_age_in_ms|
+------------+------+-------------------+-----------------------+---------------------+----------------------+
|audit_branch|BRANCH|4474315599443171786|                   NULL|                 NULL|                  NULL|
|release_2023|   TAG|3767916740125537610|                   NULL|                 NULL|                  NULL|
|        main|BRANCH|3767916740125537610|                   NULL|                 NULL|                  NULL|
+------------+------+-------------------+-----------------------+---------------------+----------------------+



In [14]:
# main is untouched by the branch write; the branch sees its own row
print("main  rows in 2024:", spark.sql(
    "SELECT count(*) FROM my_catalog.spy_table WHERE Year(Date) = 2024").collect()[0][0])
print("audit rows in 2024:", spark.sql(
    "SELECT count(*) FROM my_catalog.spy_table VERSION AS OF 'audit_branch' "
    "WHERE Year(Date) = 2024").collect()[0][0])
print("tag release_2023 total:", spark.sql(
    "SELECT count(*) FROM my_catalog.spy_table VERSION AS OF 'release_2023'").collect()[0][0])

main  rows in 2024: 0


audit rows in 2024: 1


tag release_2023 total: 7283


Branches can carry their own retention policy, and `expire_snapshots` will never
expire a snapshot that a branch or tag still points at, which makes branches a
good safety net before a maintenance window:

```sql
ALTER TABLE my_catalog.spy_table CREATE BRANCH keep_two_weeks
RETAIN 14 DAYS WITH SNAPSHOT RETENTION 7 SNAPSHOTS 2 DAYS
```

Clean up with `ALTER TABLE ... DROP BRANCH audit_branch` and
`DROP TAG release_2023`.

## 9. NEW in Spark 4.2 - one-statement upserts

Spark 4.2 folds row-level and partition-level replacement into `INSERT`, so a
corrected batch no longer needs a staging table.

In [15]:
spark.sql("DROP TABLE IF EXISTS my_catalog.upsert_demo")
spark.sql("""
CREATE TABLE my_catalog.upsert_demo (id INT, region STRING, revenue DOUBLE)
USING iceberg
PARTITIONED BY (region)
""")
spark.sql("""
INSERT INTO my_catalog.upsert_demo VALUES
  (1, 'emea', 100.0), (2, 'emea', 200.0), (3, 'apac', 300.0), (4, 'amer', 400.0)
""")

spark.sql("SELECT * FROM my_catalog.upsert_demo ORDER BY region, id").show()

+---+------+-------+
| id|region|revenue|
+---+------+-------+
|  4|  amer|  400.0|
|  3|  apac|  300.0|
|  1|  emea|  100.0|
|  2|  emea|  200.0|
+---+------+-------+



In [16]:
# Replace a whole partition in one statement.
spark.sql("""
INSERT INTO my_catalog.upsert_demo REPLACE WHERE region = 'emea'
SELECT 2 AS id, 'emea' AS region, 99.0 AS revenue
""")

# Copy-on-write needs a predicate that matches whole data files. Switching the
# table to merge-on-read makes a partial-row predicate fine:
spark.sql("ALTER TABLE my_catalog.upsert_demo "
          "SET TBLPROPERTIES ('write.delete.mode' = 'merge-on-read')")

spark.sql("""
INSERT INTO my_catalog.upsert_demo REPLACE WHERE id = 2
SELECT 2 AS id, 'emea' AS region, 99.5 AS revenue
""")

spark.sql("SELECT * FROM my_catalog.upsert_demo ORDER BY region, id").show()

+---+------+-------+
| id|region|revenue|
+---+------+-------+
|  4|  amer|  400.0|
|  3|  apac|  300.0|
|  2|  emea|   99.5|
+---+------+-------+



In [17]:
# BY NAME matches columns by name instead of position, so the incoming schema
# is free to drift in order.
spark.sql("""
INSERT INTO my_catalog.upsert_demo BY NAME REPLACE WHERE id = 3
SELECT 777.0 AS revenue, 'apac' AS region, 3 AS id
""")

spark.sql("SELECT * FROM my_catalog.upsert_demo ORDER BY region, id").show()

+---+------+-------+
| id|region|revenue|
+---+------+-------+
|  4|  amer|  400.0|
|  3|  apac|  777.0|
|  2|  emea|   99.5|
+---+------+-------+



In [18]:
# Whole-partition replacement the classic way still works too.
spark.createDataFrame(
    [(5, 'emea', 500.0), (6, 'emea', 600.0), (7, 'apac', 700.0)],
    "id INT, region STRING, revenue DOUBLE",
).createOrReplaceTempView("batch")

spark.sql("INSERT OVERWRITE my_catalog.upsert_demo SELECT * FROM batch")

spark.sql("SELECT * FROM my_catalog.upsert_demo ORDER BY region, id").show()

+---+------+-------+
| id|region|revenue|
+---+------+-------+
|  7|  apac|  700.0|
|  5|  emea|  500.0|
|  6|  emea|  600.0|
+---+------+-------+



## 10. NEW in Spark 4.2 - `QUALIFY` and vector functions

In [19]:
# QUALIFY filters on a window function without a wrapping subquery.
spark.sql("""
SELECT Date, Close,
       row_number() OVER (PARTITION BY Year(Date) ORDER BY Close DESC) AS rank_in_year
FROM my_catalog.spy_table
QUALIFY rank_in_year <= 3
ORDER BY Date
""").show(8)

+----------+---------+------------+
|      Date|    Close|rank_in_year|
+----------+---------+------------+
|1993-10-15|  47.0625|           2|
|1993-12-28| 47.09375|           1|
|1993-12-29| 47.03125|           3|
|1994-01-31| 48.21875|           2|
|1994-02-02| 48.28125|           1|
|1994-02-03|  48.0625|           3|
|1995-12-06| 62.28125|           3|
|1995-12-11|62.421875|           2|
+----------+---------+------------+
only showing top 8 rows


In [20]:
# Vector expressions (SPARK-54713): ARRAY<FLOAT> in, similarity out.
spark.sql("""
SELECT vector_norm(CAST(array(3.0, 4.0) AS ARRAY<FLOAT>)) AS norm,
       vector_l2_distance(CAST(array(0.0, 0.0) AS ARRAY<FLOAT>),
                          CAST(array(3.0, 4.0) AS ARRAY<FLOAT>)) AS l2,
       vector_cosine_similarity(CAST(array(1.0, 0.0) AS ARRAY<FLOAT>),
                                CAST(array(1.0, 1.0) AS ARRAY<FLOAT>)) AS cosine
""").show()

+----+---+----------+
|norm| l2|    cosine|
+----+---+----------+
| 5.0|5.0|0.70710677|
+----+---+----------+



In [21]:
# The same functions over an Iceberg column, so a distance-based join runs
# entirely inside Spark instead of exporting the vectors.
spark.sql("DROP TABLE IF EXISTS my_catalog.vector_demo")
spark.sql("""
CREATE TABLE my_catalog.vector_demo (id INT, embedding ARRAY<FLOAT>)
USING iceberg
""")
spark.sql("""
INSERT INTO my_catalog.vector_demo VALUES
  (1, CAST(array(1.0, 0.0) AS ARRAY<FLOAT>)),
  (2, CAST(array(0.0, 1.0) AS ARRAY<FLOAT>)),
  (3, CAST(array(0.6, 0.8) AS ARRAY<FLOAT>))
""")

spark.sql("""
SELECT a.id AS a_id, b.id AS b_id,
       round(vector_l2_distance(a.embedding, b.embedding), 4) AS distance
FROM my_catalog.vector_demo a
CROSS JOIN my_catalog.vector_demo b
WHERE a.id < b.id
ORDER BY distance
""").show()

+----+----+--------+
|a_id|b_id|distance|
+----+----+--------+
|   2|   3|  0.6325|
|   1|   3|  0.8944|
|   1|   2|  1.4142|
+----+----+--------+



## 11. NEW in Iceberg 1.12 - what changed for this setup

**The JDBC catalog needs `jdbc.schema-version=V1`.** Iceberg 1.12 gave the JDBC
catalog view support, and it now refuses *every* operation until you opt in to
the schema migration:

```text
UnsupportedOperationException: JDBC catalog is initialized without view support.
To auto-migrate the database's schema and enable view support, set jdbc.schema-version=V1
```

`spark/spark-defaults.conf` sets it. The migrated tables are visible in pgAdmin
(<http://localhost:5050>) or via:

```bash
docker compose exec db psql -U icebergcat -d icebergcat -c '\dt'
```

**Spark 3.4 support is gone.** Iceberg 1.12 targets Spark 3.5 and 4.x, which is
why the runtime artifact is `iceberg-spark-runtime-4.1_2.13`: the Spark 4.x line,
Scala 2.13.

**`iceberg-aws-bundle` is mandatory.** Since Iceberg 1.3 the AWS SDK v2 is no
longer shaded into the Spark runtime, and *neither* Iceberg artifact declares
dependencies, so `spark.jars.packages` cannot pull it in transitively. Without it
`S3FileIO` fails with `NoClassDefFoundError`.

In [22]:
# Iceberg views are the feature gated on jdbc.schema-version, and they are the
# one thing that does not work on Spark 4.2 (section 16). Everything else,
# including all table DDL, works.
def first_error_line(exc, *needles):
    lines = [ln.strip().lstrip(": ") for ln in str(exc).splitlines() if ln.strip()]
    lines = [ln for ln in lines if not ln.startswith('{"ts"')]
    for ln in lines:
        if any(n in ln for n in needles):
            return ln
    return lines[1] if len(lines) > 1 else lines[0]

try:
    spark.sql("CREATE OR REPLACE VIEW my_catalog.v_probe AS SELECT 1 AS one")
    spark.sql("SELECT one FROM my_catalog.v_probe").show()
    spark.sql("DROP VIEW IF EXISTS my_catalog.v_probe")
except Exception as exc:
    print("Iceberg views on Spark 4.2 ->",
          first_error_line(exc, "INTERNAL_ERROR", "Exception")[:150])

Iceberg views on Spark 4.2 -> org.apache.spark.SparkException: [INTERNAL_ERROR] The Spark SQL phase analysis failed with an internal error. You hit a bug in Spark or the Spark plug


## 12. Maintenance

Every Iceberg commit adds files and a new metadata file, so a table left alone
drifts towards many tiny files and thousands of snapshots. Each remedy is a
single `CALL`.

In [23]:
spark.sql("""
SELECT partition, count(*) AS files,
       sum(record_count) AS records,
       round(avg(file_size_in_bytes) / 1024, 1) AS avg_kb
FROM my_catalog.upsert_demo.files
GROUP BY partition
ORDER BY partition
""").show(truncate=40)

+---------+-----+-------+------+
|partition|files|records|avg_kb|
+---------+-----+-------+------+
|   {apac}|    1|      1|   0.9|
|   {emea}|    1|      2|   1.0|
+---------+-----+-------+------+



In [24]:
# Compact small data files, folding in position delete files as well.
spark.sql("""
CALL my_catalog.system.rewrite_data_files(
  table => 'upsert_demo',
  options => map('target-file-size-bytes', '8388608', 'min-input-files', '2'))
""").show(truncate=40)

+--------------------------+----------------------+---------------------+-----------------------+--------------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|removed_delete_files_count|
+--------------------------+----------------------+---------------------+-----------------------+--------------------------+
|                         0|                     0|                    0|                      0|                         0|
+--------------------------+----------------------+---------------------+-----------------------+--------------------------+



In [25]:
# Manifests are the scan-planning input: rewriting them drops entries for
# deleted files and groups data files by partition.
spark.sql("CALL my_catalog.system.rewrite_manifests(table => 'upsert_demo')").show(truncate=40)

spark.sql("CALL my_catalog.system.rewrite_position_delete_files(table => 'spy_table')").show(truncate=40)

26/10/02 12:19:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------------------------+---------------------+
|rewritten_manifests_count|added_manifests_count|
+-------------------------+---------------------+
|                        4|                    1|
+-------------------------+---------------------+

+----------------------------+------------------------+---------------------+-----------------+
|rewritten_delete_files_count|added_delete_files_count|rewritten_bytes_count|added_bytes_count|
+----------------------------+------------------------+---------------------+-----------------+
|                           0|                       0|                    0|                0|
+----------------------------+------------------------+---------------------+-----------------+



In [26]:
# Drop old snapshots and the files no longer referenced by any of them.
spark.sql("""
CALL my_catalog.system.expire_snapshots(
  table => 'spy_table',
  retain_last => 2,
  older_than => TIMESTAMP '2000-01-01 00:00:00')
""").show(truncate=40)

+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|deleted_data_files_count|deleted_position_delete_files_count|deleted_equality_delete_files_count|deleted_manifest_files_count|deleted_manifest_lists_count|deleted_statistics_files_count|
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|                       0|                                  0|                                  0|                           0|                           0|                             0|
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+



In [27]:
# Anything sitting in the table location that no snapshot points at. dry_run
# lists candidates without deleting. prefix_listing=true routes the directory
# walk through Iceberg's FileIO; without it this procedure needs a Hadoop
# filesystem implementation registered for s3://.
spark.sql("""
CALL my_catalog.system.remove_orphan_files(
  table => 'spy_table',
  older_than => TIMESTAMP '2000-01-01 00:00:00',
  dry_run => true,
  prefix_listing => true)
""").show(truncate=80)

+--------------------+
|orphan_file_location|
+--------------------+
+--------------------+



## 13. Rollback

`rollback_to_snapshot` rewinds the table's *current* pointer. It never deletes
anything, so the snapshot you were on is still addressable afterwards. We do it
on `upsert_demo` so the main table keeps its data.

In [28]:
spark.sql("""
SELECT snapshot_id, operation, summary['total-records'] AS total_records
FROM my_catalog.upsert_demo.snapshots
ORDER BY committed_at
""").show(truncate=40)

+-------------------+---------+-------------+
|        snapshot_id|operation|total_records|
+-------------------+---------+-------------+
| 642485710533828834|   append|            4|
|2376806145374324691|overwrite|            3|
|5424191287917874415|overwrite|            3|
| 770368010853880026|overwrite|            3|
|3337509105018373667|overwrite|            3|
|9037944339050637295|  replace|            3|
+-------------------+---------+-------------+



In [29]:
upsert_snapshots = spark.sql(
    "SELECT snapshot_id FROM my_catalog.upsert_demo.snapshots ORDER BY committed_at"
).collect()
target = upsert_snapshots[0]["snapshot_id"]

print("current rows:", spark.sql("SELECT count(*) FROM my_catalog.upsert_demo").collect()[0][0])
spark.sql("CALL my_catalog.system.rollback_to_snapshot("
          "table => 'upsert_demo', snapshot_id => " + str(target) + ")")
print("after rollback:", spark.sql("SELECT count(*) FROM my_catalog.upsert_demo").collect()[0][0])

spark.sql("SELECT * FROM my_catalog.upsert_demo ORDER BY region, id").show()

current rows: 3


after rollback: 4
+---+------+-------+
| id|region|revenue|
+---+------+-------+
|  4|  amer|  400.0|
|  3|  apac|  300.0|
|  1|  emea|  100.0|
|  2|  emea|  200.0|
+---+------+-------+



The rollback itself is just a metadata pointer move. By timestamp it looks like:

```sql
CALL my_catalog.system.rollback_to_timestamp(
  table => 'upsert_demo', timestamp => TIMESTAMP '2022-07-25 12:15:00')
```

Note the contrast with `expire_snapshots` in section 12: rollback moves the
pointer, expiry deletes history. Neither undoes the other.

## 14. Metadata tables

The table format makes its own internals queryable as plain SQL, and it is the
fastest way to understand what a commit actually did.

In [30]:
spark.sql("""
SELECT h.made_current_at, s.operation, h.snapshot_id, h.is_current_ancestor,
       s.summary['added-data-files']   AS added_files,
       s.summary['added-records']     AS added_rows,
       s.summary['total-data-files']  AS total_files
FROM my_catalog.upsert_demo.history h
JOIN my_catalog.upsert_demo.snapshots s ON h.snapshot_id = s.snapshot_id
ORDER BY made_current_at
""").show(truncate=60)

+-----------------------+---------+-------------------+-------------------+-----------+----------+-----------+
|        made_current_at|operation|        snapshot_id|is_current_ancestor|added_files|added_rows|total_files|
+-----------------------+---------+-------------------+-------------------+-----------+----------+-----------+
|2026-10-02 12:19:22.212|   append| 642485710533828834|               true|          3|         4|          3|
|2026-10-02 12:19:22.422|overwrite|2376806145374324691|              false|          1|         1|          3|
|2026-10-02 12:19:22.593|overwrite|5424191287917874415|              false|          1|         1|          3|
| 2026-10-02 12:19:22.78|overwrite| 770368010853880026|              false|          1|         1|          3|
|2026-10-02 12:19:23.644|overwrite|3337509105018373667|              false|          2|         3|          2|
|2026-10-02 12:19:25.207|  replace|9037944339050637295|              false|       NULL|      NULL|          2|
|

In [31]:
spark.sql("SELECT * FROM my_catalog.spy_table.partitions").show(truncate=40)

spark.sql("""
SELECT content, length, partition_spec_id, added_snapshot_id
FROM my_catalog.spy_table.manifests
ORDER BY content, added_snapshot_id
""").show(truncate=40)

+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|partition|spec_id|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|        last_updated_at|last_updated_snapshot_id|
+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|      {8}|      1|         422|         1|                        12102|                          21|                         1|                           0|                         0|2026-10-02 12:19:21.159|     3767916740125537610|
|      {9}|      1|         389|         1|                 

In [32]:
spark.sql("""
SELECT file_path, record_count, spec_id, content
FROM my_catalog.spy_table.all_data_files
ORDER BY file_path
LIMIT 6
""").show(truncate=70)

+----------------------------------------------------------------------+------------+-------+-------+
|                                                             file_path|record_count|spec_id|content|
+----------------------------------------------------------------------+------------+-------+-------+
|s3://iceberg/spy_data/data/00000-5-58c37a37-fecd-4b9b-9000-3199199f...|        2753|      0|      0|
|s3://iceberg/spy_data/data/00000-8-c4271bfb-d69f-466d-b956-c9df7eae...|        2753|      0|      0|
|s3://iceberg/spy_data/data/Month=1/00000-10-017f903e-93c6-4d20-8819...|         403|      1|      0|
|s3://iceberg/spy_data/data/Month=1/00000-40-1ca875b8-68aa-44ec-81ab...|         383|      1|      0|
|s3://iceberg/spy_data/data/Month=1/00000-49-110b2ee3-c61e-4dad-818e...|         383|      1|      0|
|s3://iceberg/spy_data/data/Month=1/00000-71-5ff8b1ca-0ded-457a-b5d4...|           1|      1|      0|
+----------------------------------------------------------------------+----------

## 15. SeaweedFS's own Iceberg REST catalog

SeaweedFS does more than store bytes: it serves the Iceberg REST catalog on
`:8181`. Each S3 Tables bucket is one catalog, selected through the `warehouse`
property (`spark.sql.catalog.seaweed_catalog.warehouse=s3://iceberg-catalog`).

This is the piece that lets other engines read the data with no shared Postgres:

| Engine | Auth |
|---|---|
| Spark, Trino, RisingWave | SigV4 |
| DuckDB, Doris | OAuth2 token endpoint |
| Dremio | unsigned REST + S3 keys |

In [33]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS seaweed_catalog.demo")
spark.sql("DROP TABLE IF EXISTS seaweed_catalog.demo.events")

spark.sql("""
CREATE TABLE seaweed_catalog.demo.events (
  event_id INT, user_id INT, kind STRING, ts TIMESTAMP)
USING iceberg
PARTITIONED BY (days(ts))
""")

spark.sql("""
INSERT INTO seaweed_catalog.demo.events VALUES
  (1, 101, 'click', TIMESTAMP '2026-10-01 10:00:00'),
  (2, 101, 'view',  TIMESTAMP '2026-10-01 11:30:00'),
  (3, 102, 'click', TIMESTAMP '2026-10-02 09:15:00')
""")

spark.sql("SELECT * FROM seaweed_catalog.demo.events ORDER BY event_id").show()

26/10/02 12:19:28 WARN AuthManagers: The property rest.sigv4-enabled is deprecated and will be removed in a future release. Please use the property rest.auth.type=sigv4 instead.


+--------+-------+-----+-------------------+
|event_id|user_id| kind|                 ts|
+--------+-------+-----+-------------------+
|       1|    101|click|2026-10-01 10:00:00|
|       2|    101| view|2026-10-01 11:30:00|
|       3|    102|click|2026-10-02 09:15:00|
+--------+-------+-----+-------------------+



In [34]:
spark.sql("SHOW NAMESPACES IN seaweed_catalog").show()
spark.sql("SHOW TABLES IN seaweed_catalog.demo").show()

spark.sql("""
SELECT file_path, partition, record_count, file_format
FROM seaweed_catalog.demo.events.files
""").show(truncate=70)

+---------+
|namespace|
+---------+
|     demo|
+---------+

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|     demo|   events|      false|
+---------+---------+-----------+

+----------------------------------------------------------------------+------------+------------+-----------+
|                                                             file_path|   partition|record_count|file_format|
+----------------------------------------------------------------------+------------+------------+-----------+
|s3://iceberg-catalog/demo/events/data/ts_day=2026-10-02/00000-240-0...|{2026-10-02}|           1|    PARQUET|
|s3://iceberg-catalog/demo/events/data/ts_day=2026-10-01/00000-240-0...|{2026-10-01}|           2|    PARQUET|
+----------------------------------------------------------------------+------------+------------+-----------+



In [35]:
# The procedure API is identical over REST, so maintenance is catalog-agnostic.
spark.sql("CALL seaweed_catalog.system.rewrite_manifests(table => 'demo.events')").show(truncate=40)

spark.sql("""
SELECT snapshot_id, operation FROM seaweed_catalog.demo.events.snapshots
""").show(truncate=40)

+-------------------------+---------------------+
|rewritten_manifests_count|added_manifests_count|
+-------------------------+---------------------+
|                        0|                    0|
+-------------------------+---------------------+

+-------------------+---------+
|        snapshot_id|operation|
+-------------------+---------+
|4334779716112073393|   append|
+-------------------+---------+



26/10/02 12:19:29 WARN Spark3Util: Failed to load catalog: demo
org.apache.spark.sql.connector.catalog.CatalogNotFoundException: [CATALOG_NOT_FOUND] The catalog `demo` not found. Consider to set the SQL config "spark.sql.catalog.demo" to a catalog plugin. SQLSTATE: 42P08
	at org.apache.spark.sql.errors.QueryExecutionErrors$.catalogNotFoundError(QueryExecutionErrors.scala:1897)
	at org.apache.spark.sql.connector.catalog.Catalogs$.load(Catalogs.scala:57)
	at org.apache.spark.sql.connector.catalog.DefaultCatalogManager.$anonfun$catalog$1(CatalogManager.scala:134)
	at scala.collection.mutable.HashMap.getOrElseUpdate(HashMap.scala:469)
	at org.apache.spark.sql.connector.catalog.DefaultCatalogManager.catalog(CatalogManager.scala:134)
	at org.apache.iceberg.spark.Spark3Util.lambda$catalogAndIdentifier$5(Spark3Util.java:845)
	at org.apache.iceberg.spark.SparkUtil.catalogAndIdentifier(SparkUtil.java:128)
	at org.apache.iceberg.spark.Spark3Util.catalogAndIdentifier(Spark3Util.java:841)
	at org.a

## 16. Gaps you will hit on this stack

Worth knowing before you file a bug.

**1. Iceberg views do not work on Spark 4.2.** `CREATE VIEW` on an Iceberg
catalog fails with an internal error, because Spark 4.2 turned
`connector.catalog.View` from an interface into a class while Iceberg's view
implementation still targets the old shape. This affects every Iceberg version
currently published, so it is not fixed by moving off the 1.12.0 snapshot.

**2. There is no released Iceberg artifact for Spark 4.2.** Iceberg 1.12.0
publishes `iceberg-spark-runtime-{3.5,4.0,4.1}_2.13` only. The 4.1 runtime
cannot run on Spark 4.2 for the same `View` reason, so this repo pins the
era-matched `1.12.0-SNAPSHOT` pair and resolves them from the Apache snapshot
repository via `spark/ivysettings.xml`. See "Switching to released artifacts"
in the README if you prefer released jars.

**3. `TABLESAMPLE SYSTEM` parses but not against Iceberg.** Spark 4.2 added
block sampling with DSv2 pushdown, but Iceberg's scan does not implement
block-level sampling, so you get `UNSUPPORTED_FEATURE.TABLESAMPLE_SYSTEM`.

**4. `INSERT ... WITH SCHEMA EVOLUTION` and `REPLACE ON/USING` are not in Spark
4.2.0's SQL parser.** Both appear in the 4.2.0 release notes but neither is in
the 4.2.0 SQL reference, and both fail to parse here. `INSERT INTO ... REPLACE
WHERE` and `BY NAME REPLACE WHERE`, used in section 9, are the released forms.

**5. Geospatial `GEOMETRY`/`GEOGRAPHY` types are not in the 4.2.0 classic SQL
parser either.** `spark.sql.geospatial.enabled` exists and the 4.2.0 data-type
reference documents the types, but `CAST('POINT(0 0)' AS GEOMETRY)` and every
`ST_*` function are absent from `SHOW FUNCTIONS`.

**6. `remove_orphan_files` needs `prefix_listing => true`.** The default
directory walk goes through Hadoop's `FileSystem`, which has no `s3://`
implementation registered in this image. Iceberg's FileIO does, so the flag is
the fix.

**7. `SHOW TABLES IN <jdbc-catalog>` with no namespace fails.** The JDBC catalog
has no `default` namespace, so the unqualified form raises
`NoSuchNamespaceException`. Use `SHOW NAMESPACES`, or qualify the namespace.

In [36]:
# Demonstrate gaps 3 and 5 rather than just claiming them.
for label, sql in [
    ("TABLESAMPLE against Iceberg",
     "SELECT * FROM my_catalog.spy_table TABLESAMPLE SYSTEM (10 PERCENT)"),
    ("GEOMETRY type",
     "SELECT CAST('POINT(0 0)' AS GEOMETRY) AS g"),
]:
    try:
        spark.sql(sql).show()
        print(label + ": unexpectedly succeeded")
    except Exception as exc:
        print(label + " ->",
              first_error_line(exc, "UNSUPPORTED_FEATURE", "UNSUPPORTED_DATATYPE")[:150])

TABLESAMPLE against Iceberg -> [UNSUPPORTED_FEATURE.TABLESAMPLE_SYSTEM] The feature is not supported: TABLESAMPLE SYSTEM is only supported by data sources that implement block-level


{"ts": "2026-10-02 12:19:29.961", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[UNSUPPORTED_DATATYPE] Unsupported data type \"GEOMETRY\". SQLSTATE: 0A000", "context": {"errorClass": "UNSUPPORTED_DATATYPE"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o69.sql.\n: org.apache.spark.sql.catalyst.parser.ParseException: \n[UNSUPPORTED_DATATYPE] Unsupported data type \"GEOMETRY\". SQLSTATE: 0A000\n== SQL (line 1, position 29) ==\nSELECT CAST('POINT(0 0)' AS GEOMETRY) AS g\n                            ^^^^^^^^\n\n\tat org.apache.spark.sql.errors.QueryParsingErrors$.dataTypeUnsupportedError(QueryParsingErrors.scala:393)\n\tat org.apache.spark.sql.catalyst.parser.DataTypeAstBuilder.$anonfun$visitPrimitiveDataType$1(DataTypeAstBuilder.scala:417)\n\tat org.apache.spark.sql.catalyst.util.SparkParserUtils.withOrigin(SparkParserUtils.scala:221)\n\tat org.apache.spark.sql.catalyst.util.SparkParserUtils.withOrigin$(SparkParserUtils.scala:204)\n\tat or

GEOMETRY type -> [UNSUPPORTED_DATATYPE] Unsupported data type "GEOMETRY". SQLSTATE: 0A000


## 17. Clean up

In [37]:
for table in ("spy_table", "upsert_demo", "vector_demo", "spy_data"):
    spark.sql("DROP TABLE IF EXISTS my_catalog." + table)

# v_probe is a view, and Spark refuses DROP TABLE on a view.
spark.sql("DROP VIEW IF EXISTS my_catalog.v_probe")

spark.sql("DROP TABLE IF EXISTS seaweed_catalog.demo.events")
spark.sql("DROP NAMESPACE IF EXISTS seaweed_catalog.demo")

# NOTE: SHOW TABLES IN <catalog> with no namespace fails on the JDBC catalog,
# because Iceberg's JDBC catalog has no `default` namespace. List namespaces,
# or qualify the namespace explicitly.
spark.sql("SHOW NAMESPACES IN my_catalog").show()
spark.sql("SHOW NAMESPACES IN seaweed_catalog").show()

+---------+
|namespace|
+---------+
+---------+

+---------+
|namespace|
+---------+
+---------+



To drop everything, including the object store contents and the catalog tables:

```bash
docker compose down -v
rm -rf warehouse/* && touch warehouse/.gitkeep
```

## Reference

- [Apache Iceberg 1.12.0 docs](https://iceberg.apache.org/docs/latest/)
- [Spark Iceberg integration on SeaweedFS](https://github.com/seaweedfs/seaweedfs/wiki/Spark-Iceberg-Integration)
- [SeaweedFS Iceberg catalog](https://github.com/seaweedfs/seaweedfs/wiki/SeaweedFS-Iceberg-Catalog)
- [Spark 4.2.0 release notes](https://spark.apache.org/releases/spark-release-4-2-0.html)